# Lab 2: Graph for expense approval

An employee has claimed USD 4,850 for conference travel. In this lab a Strands `Graph` decides the claim. An intake node opens it, a policy check and a receipt check run side by side, and a reviewer joins their results into a typed decision. The claim is over the USD 2,500 limit, so the graph routes it to a manager-approval node, which stops the graph until a person answers.

That answer can take hours, so the graph keeps its state in Amazon S3 through the Strands `S3SessionManager`. A new graph instance, built later with the same session ID, carries on where the first one stopped.

## What you will do

1. Run the graph until it stops for the manager.
2. Read the evidence: execution order and timings, the results that reached the join, the interrupt, and the objects in S3.
3. Build a fresh graph, give it the manager's answer, and check what it restored and what it ran.
4. Route the manager's answer on text, then on a validated status, and compare what each one pays.

**Estimated time:** 20-25 minutes.

You do not change any code. Run the cells from top to bottom and compare each output with its **What to look for** list.

![Intake runs first, then a policy check and a receipt check run concurrently. A reviewer joins both results, and a condition routes an over-limit claim to manager approval, whose state is saved to Amazon S3.](images/topology.png)


## Why this is a Graph

Expense approval has a route you can write down before any claim arrives: open the claim, check policy and receipts, decide, and ask a manager only when the rules say so. In a Graph your code owns that route. Nodes are Agents, edges say what runs after what, and an optional condition on an edge decides at run time whether it is followed. The models read and judge inside their nodes, but none of them picks the next step.

Nodes that become ready together run as one batch, concurrently, so the two checks overlap and both results reach `review`.

The manager-approval Agent calls a tool that uses `tool_context.interrupt()`. The call pauses that Agent and returns a graph result with status `INTERRUPTED`. The result includes the question for the manager and an interrupt ID. The manager's answer must use that ID so Strands can resume the correct call.

The `S3SessionManager` belongs to the Graph. It saves the completed node results and the interrupt state after each step. A node Agent must not have its own session manager, because the Graph owns the shared execution state. Later, a fresh Graph with the same session ID restores that state and continues from the interrupted node.

## Setup

Run the `%pip` cell by itself first. If it installs or upgrades anything, restart the kernel. Then select the next cell and choose **Run > Run All Below**. Starting **Run All** above the install cell can stop after a package-triggered kernel restart.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import json
import sys
import time
import uuid
from pathlib import Path
from typing import Literal

import boto3
from pydantic import BaseModel, Field
from strands import Agent, ToolContext, tool
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder, Status
from strands.session import S3SessionManager

sys.path.insert(0, str(Path("..").resolve()))
from utils.live_panes import LivePanes
from utils.stacks import cleanup, find_or_deploy
from utils.validation import write_validation_artifact

The intake, the checks, and the payment step run on Amazon Nova Pro. The reviewer and the manager-approval node run on Anthropic Claude Sonnet 5. Both models are called in the AWS Region of this notebook's session.

In [ ]:
region = boto3.Session().region_name
nova = BedrockModel(model_id="us.amazon.nova-pro-v1:0", region_name=region)
sonnet = BedrockModel(model_id="global.anthropic.claude-sonnet-5", region_name=region)

### Lab resources

The lab stack holds one S3 bucket for graph sessions. At an event the stack already exists. On your own account, the next cell deploys it, which takes about a minute.

In [ ]:
outputs = find_or_deploy("graph", "infra/graph.yaml")
bucket = outputs["SessionBucketName"]
s3 = boto3.client("s3", region_name=region)  # to read what the session manager writes

## The claim, the reviewer's decision, and the approval tool

The claim and the policy are small inline values, and the task hands the whole claim to the entry node.

The reviewer returns a `ReviewDecision`, declared with `structured_output_model`. Strands validates the answer against the schema, and a `status` outside the `Literal` fails validation and is sent back to the model. The edge condition `needs_manager` then compares a validated field. Strands also evaluates conditions when it saves the session, sometimes before `review` has run, so the condition checks that the result exists.

`request_manager_approval` is the only tool in the graph. The first time it runs, `tool_context.interrupt()` raises, and the graph stops. When the graph resumes with an answer for that interrupt ID, Strands runs the tool again from the top, and this time `interrupt()` returns the answer.

`join_inputs()` reads a joining Agent's conversation and lists, for each of its turns, which nodes' results were in its input.

In [ ]:
# The claim and the expense policy.
CLAIM = {
    "claim_id": "EXP-2041",
    "employee": "Dana Kim",
    "category": "conference travel",
    "amount_usd": 4850.00,
    "purpose": "Speaking at the Northwest Data Engineering Summit",
    "receipts": [
        {
            "merchant": "Harbor Hotel",
            "amount_usd": 2100.00,
        },
        {
            "merchant": "Cascade Air",
            "amount_usd": 1450.00,
        },
        {
            "merchant": "Summit registration",
            "amount_usd": 1300.00,
        },
    ],
}
POLICY = (
    "Conference travel is reimbursed up to USD 5,000 per trip. A claim over USD 2,500 needs the "
    "employee's manager to approve it. Every item over USD 75 needs a receipt."
)

# The graph's task, and the node events for the timeline.
task = f"Decide expense claim {CLAIM['claim_id']}: {json.dumps(CLAIM)}"  # one string; the claim as JSON
# The "type" values of the events the graph yields when a node starts, stops, or is interrupted.
NODE_EVENTS = ("multiagent_node_start", "multiagent_node_stop", "multiagent_node_interrupt")


# The reviewer's typed decision.
class ReviewDecision(BaseModel):
    """The reviewer's decision. The graph routes on `status`."""

    status: Literal["approve", "needs_manager", "reject"] = Field(description="The decision on the claim")
    reason: str = Field(description="One sentence naming the rule behind the decision")


# The edge condition: route to the manager on a validated status.
def needs_manager(state):
    """Edge condition from review to manager_approval."""
    # The graph passes its GraphState; state.results maps each finished node ID to its NodeResult.
    if "review" not in state.results:
        return False
    # .result is the node's AgentResult; structured_output is the validated ReviewDecision.
    decision = state.results["review"].result.structured_output  # None on a result restored from S3
    return decision is not None and decision.status == "needs_manager"


# The approval tool: it raises the interrupt that stops the graph.
# context=True passes a ToolContext as tool_context; the model sees only the other parameters.
@tool(context=True)
def request_manager_approval(claim_id: str, amount_usd: float, summary: str, tool_context: ToolContext) -> str:
    """Ask the employee's manager to approve an over-limit expense claim, and return the manager's answer."""
    # The first call raises an interrupt with this name and reason, which stops the Agent and the graph.
    # When the graph resumes with a response for it, the tool runs again and interrupt() returns the response.
    answer = tool_context.interrupt(
        "manager_approval",
        reason={
            "claim_id": claim_id,
            "amount_usd": amount_usd,
            "summary": summary,
        },
    )
    return str(answer)


# A helper that shows which results reached a joining node.
def join_inputs(agent):
    """For each turn of a joining Agent, the node IDs whose results were in its input."""
    # A node with incoming results gets text blocks: "Original Task: ...", then "\nFrom <node_id>:" per result.
    # The slice cuts "From " and the trailing colon, which leaves the node ID.
    return [
        [
            block["text"].strip()[len("From ") : -1]
            for block in message["content"]
            if block.get("text", "").startswith("\nFrom ")
        ]
        for message in agent.messages
        if message["role"] == "user" and message["content"][0].get("text", "").startswith("Original Task")
    ]

## What to look for

The graph starts at `intake` with a new session ID.

Look for:

- Which nodes run in the same batch?
- Which nodes' results reach `review`?
- Which node stops the graph, and what is `status` then?
- Which objects does the session manager write to S3?


## Run the graph until it asks the manager

The cell defines the five Agents and builds the graph with `GraphBuilder`. The session manager goes on the builder, next to the limits that bound the run. The loop sends every event of `graph.stream_async(task)` to `panes.show()` and records when each node starts and stops. The last event holds the `GraphResult`.

Every Agent sets `callback_handler=None`: `stream_async` still calls each Agent's default handler, which would print the same text a second time below the panes.

**What to look for:** `policy_check` and `receipt_check` running at the same time, `[tool: request_manager_approval]` in the `manager_approval` pane, and that pane ending as `Interrupted`.

In [ ]:
# One session per approval request; the graph saves state under it.
session_id = f"claim-{CLAIM['claim_id']}-{uuid.uuid4().hex[:8]}"

# The nodes: Nova Pro checks the claim, Sonnet 5 reviews it.
intake = Agent(
    name="intake",
    model=nova,
    system_prompt="You open an expense claim. Summarize it in three short lines: who, what, and how much.",
    callback_handler=None,
)

policy_check = Agent(
    name="policy_check",
    model=nova,
    system_prompt=(
        f"You check an expense claim against this policy: {POLICY} "
        "Say which rules apply and whether the claim meets each one. Answer in at most four sentences."
    ),
    callback_handler=None,
)

receipt_check = Agent(
    name="receipt_check",
    model=nova,
    system_prompt=(
        "You check that the receipts of an expense claim add up to the claimed amount. "
        "Show the sum. Answer in at most three sentences."
    ),
    callback_handler=None,
)

review = Agent(
    name="review",
    model=sonnet,
    structured_output_model=ReviewDecision,  # the typed decision the edge condition reads
    system_prompt=(
        f"You decide an expense claim from the checks you receive. Policy: {POLICY} "
        "Choose needs_manager when the policy requires the manager's approval, reject when a check failed, "
        "and approve otherwise."
    ),
    callback_handler=None,
)

manager_approval = Agent(
    name="manager_approval",
    model=sonnet,
    tools=[request_manager_approval],
    system_prompt=(
        "You get the manager's decision on an expense claim. Call request_manager_approval once with the claim "
        "ID, the amount, and a two-sentence summary of the review. Then report the manager's decision and note "
        "in at most two sentences."
    ),
    callback_handler=None,
)

# The graph: register each Agent as a node, under a node ID.
builder = GraphBuilder()
builder.add_node(intake, "intake")
builder.add_node(policy_check, "policy_check")
builder.add_node(receipt_check, "receipt_check")
builder.add_node(review, "review")
builder.add_node(manager_approval, "manager_approval")

# The edges: fan out, join at review, route on a condition.
builder.add_edge("intake", "policy_check")  # fan-out: both checks wait only for intake
builder.add_edge("intake", "receipt_check")
builder.add_edge("policy_check", "review")  # the join
builder.add_edge("receipt_check", "review")
builder.add_edge("review", "manager_approval", condition=needs_manager)  # followed only when it returns True
builder.set_entry_point("intake")  # the node that receives the task

# Persistence and limits: save state to S3, and bound the run.
builder.set_graph_id("expense_approval")  # the graph's key inside the session
# The session manager writes the graph state to S3 after each node and at the end of each call.
builder.set_session_manager(S3SessionManager(session_id=session_id, bucket=bucket, region_name=region))
builder.set_max_node_executions(10)  # node runs in total, across the whole run
builder.set_execution_timeout(600)  # seconds for the whole run
builder.set_node_timeout(180)  # seconds for each node
graph = builder.build()

# Run the graph, streaming each node into its pane.
timeline = []
started = time.perf_counter()
panes = LivePanes()
async for event in graph.stream_async(task):
    panes.show(event)
    if event.get("type") in NODE_EVENTS:
        timeline.append((time.perf_counter() - started, event["type"], event["node_id"]))  # (seconds, type, node)
result = event["result"]  # the last event carries the GraphResult

## Inspect the evidence

`execution_order` lists the nodes that finished, in order, and the timeline shows which of them overlapped. `join_inputs(review)` shows whose results reached the join. `result.interrupts` holds what the manager needs to see, and its `id` is what the answer has to quote. The last lines list the objects under the session's S3 prefix and a few fields of the saved graph state. The key layout `session_<id>/multi_agents/multi_agent_<graph id>/` is how `S3SessionManager` names its objects.

Compare this output with the questions under **What to look for**.

In [ ]:
# The run's status, order, and node timeline.
print(f"status          : {result.status}")
print(f"execution_order : {' -> '.join(node.node_id for node in result.execution_order)}")
print(
    f"nodes           : {result.completed_nodes} completed, {result.interrupted_nodes} interrupted, "
    f"{result.total_nodes} in the graph"
)
print("\ntimeline")
for seconds, kind, node_id in timeline:
    print(f"  {seconds:5.1f}s  {kind.removeprefix('multiagent_node_'):<9}  {node_id}")

# What reached the join, and the reviewer's decision.
decision = result.results["review"].result.structured_output
print(f"\nresults that reached review : {join_inputs(review)}")
print(f"review decision             : {decision.status} ({decision.reason})")

# The interrupt, and what your app would send the manager.
interrupt = result.interrupts[0]  # raised by request_manager_approval inside manager_approval
approval_request = {  # what your app would send the manager
    "interrupt_id": interrupt.id,  # needed later to answer this interrupt
    **interrupt.reason,  # claim_id, amount_usd, and summary, as the tool passed them
}
print(f"\ninterrupt {interrupt.name!r}")
print(f"  id     : {interrupt.id}")
print(f"  reason : {interrupt.reason}")

# The graph state saved in S3.
# S3SessionManager keeps everything for one session under session_<session_id>/.
prefix = f"session_{session_id}/"
print(f"\nS3 objects under s3://{bucket}/{prefix}")
for item in s3.list_objects_v2(Bucket=bucket, Prefix=prefix)["Contents"]:
    print(f"  {item['Key'].removeprefix(prefix)}  ({item['Size']} bytes)")
# The graph state lives at multi_agents/multi_agent_<graph_id>/multi_agent.json.
graph_key = f"{prefix}multi_agents/multi_agent_expense_approval/multi_agent.json"
saved = json.loads(s3.get_object(Bucket=bucket, Key=graph_key)["Body"].read())
for key in ("status", "completed_nodes", "interrupted_nodes", "next_nodes_to_execute"):
    value = saved[key] if isinstance(saved[key], str) else sorted(saved[key])  # status is a string, the rest lists
    print(f"  {key:<22}: {value}")

# Checks.
assert result.status == Status.INTERRUPTED, f"status: {result.status}"
assert decision.status == "needs_manager", f"the reviewer chose {decision.status}; rerun the cell above"
assert [sorted(sources) for sources in join_inputs(review)] == [["policy_check", "receipt_check"]], (
    "review did not run once with both checks"
)
assert saved["interrupted_nodes"] == ["manager_approval"], "the saved state has no interrupted node"

## Resume in a fresh graph instance

Nothing from the first graph is reused below: new Agents, a new `S3SessionManager`, and a new graph. Only `session_id` is the same, as it would be in another process an hour later. `builder.build()` asks the session manager for the saved state and restores it, so the cell can print the restored state before anything runs.

Look for:

- Which nodes run after the manager answers?
- Does the restored `review` result still carry its `structured_output`?
- Where does the new `manager_approval` Agent get its conversation from?


In [ ]:
# A new process: new Agents, a new session manager, and a new graph. Only session_id is reused.
intake = Agent(
    name="intake",
    model=nova,
    system_prompt="You open an expense claim. Summarize it in three short lines: who, what, and how much.",
    callback_handler=None,
)

policy_check = Agent(
    name="policy_check",
    model=nova,
    system_prompt=(
        f"You check an expense claim against this policy: {POLICY} "
        "Say which rules apply and whether the claim meets each one. Answer in at most four sentences."
    ),
    callback_handler=None,
)

receipt_check = Agent(
    name="receipt_check",
    model=nova,
    system_prompt=(
        "You check that the receipts of an expense claim add up to the claimed amount. "
        "Show the sum. Answer in at most three sentences."
    ),
    callback_handler=None,
)

review = Agent(
    name="review",
    model=sonnet,
    structured_output_model=ReviewDecision,  # the typed decision the edge condition reads
    system_prompt=(
        f"You decide an expense claim from the checks you receive. Policy: {POLICY} "
        "Choose needs_manager when the policy requires the manager's approval, reject when a check failed, "
        "and approve otherwise."
    ),
    callback_handler=None,
)

manager_approval = Agent(
    name="manager_approval",
    model=sonnet,
    tools=[request_manager_approval],
    system_prompt=(
        "You get the manager's decision on an expense claim. Call request_manager_approval once with the claim "
        "ID, the amount, and a two-sentence summary of the review. Then report the manager's decision and note "
        "in at most two sentences."
    ),
    callback_handler=None,
)

# The graph: register each Agent as a node.
builder = GraphBuilder()
builder.add_node(intake, "intake")
builder.add_node(policy_check, "policy_check")
builder.add_node(receipt_check, "receipt_check")
builder.add_node(review, "review")
builder.add_node(manager_approval, "manager_approval")

# The edges: the same fan-out, join, and condition as the first run.
builder.add_edge("intake", "policy_check")  # fan-out: both checks wait only for intake
builder.add_edge("intake", "receipt_check")
builder.add_edge("policy_check", "review")  # the join
builder.add_edge("receipt_check", "review")
builder.add_edge("review", "manager_approval", condition=needs_manager)  # followed only when it returns True
builder.set_entry_point("intake")

# Persistence and limits: the same session, so build() restores it.
builder.set_graph_id("expense_approval")  # the graph's key inside the session
builder.set_session_manager(S3SessionManager(session_id=session_id, bucket=bucket, region_name=region))
builder.set_max_node_executions(10)  # the restored count of node runs counts toward this limit
builder.set_execution_timeout(600)
builder.set_node_timeout(180)
resumed_graph = builder.build()  # build() reads the saved state from S3

# What came back from S3.
restored = resumed_graph.state
# A restored AgentResult keeps the message text, but not the structured output.
review_restored = restored.results["review"].result
print(f"status             : {restored.status}")
print(f"completed          : {sorted(node.node_id for node in restored.completed_nodes)}")
print(f"interrupted        : {sorted(node.node_id for node in restored.interrupted_nodes)}")
print(f"results            : {sorted(restored.results)}")
print(f"execution_order    : {[node.node_id for node in restored.execution_order]}")
print(f"review, as text    : {' '.join(str(review_restored).split())[:100]}...")
print(f"review, structured : {review_restored.structured_output}")
# The interrupted node gets its saved conversation back only when it resumes, so this Agent is still empty.
print(f"manager_approval messages before the answer: {len(manager_approval.messages)}")

# Checks.
assert restored.status == Status.INTERRUPTED, f"nothing was restored for session {session_id}"

The next cell answers the interrupt. A response is a list of `interruptResponse` blocks, each with an interrupt ID and any JSON value. Here the manager approves, quoting the ID from the approval request.

**What to look for:** one pane, `manager_approval`, and in it the tool call and the decision.

In [ ]:
manager_answer = "Approved. The summit is in the team's training budget."
# One interruptResponse block per pending interrupt: its ID and the answer interrupt() will return.
responses = [
    {
        "interruptResponse": {
            "interruptId": approval_request["interrupt_id"],
            "response": manager_answer,
        },
    },
]

# Passing the responses instead of a task resumes the graph from its restored state.
resume_timeline = []
started = time.perf_counter()
panes = LivePanes()
async for event in resumed_graph.stream_async(responses):
    panes.show(event)
    if event.get("type") in NODE_EVENTS:
        resume_timeline.append((time.perf_counter() - started, event["type"], event["node_id"]))
resumed_result = event["result"]

### What came back, and what ran

The first lines compare this invocation with the whole run. The conversation lines show the new `manager_approval` Agent now holds the first instance's conversation. The last lines read the saved state again.

In [ ]:
# What ran in the resumed call.
ran_now = [node_id for _, kind, node_id in resume_timeline if kind == "multiagent_node_start"]  # started this call
print(f"status             : {resumed_result.status}")
print(f"ran in this call   : {ran_now}")
print(f"execution_order    : {' -> '.join(node.node_id for node in resumed_result.execution_order)}")
print(f"execution_count    : {resumed_result.execution_count} (restored count plus this call)")
print(f"tokens             : {resumed_result.accumulated_usage['totalTokens']} (restored total plus this call)")
print(f"manager's decision : {' '.join(str(resumed_result.results['manager_approval'].result).split())[:160]}")

# The manager node's conversation after the resume: the saved one, which starts with its input from the first run.
first = manager_approval.messages[0]["content"][0]["text"]
print(f"\nmanager_approval messages now: {len(manager_approval.messages)}; the first one starts {first[:40]!r}")

# The graph state saved in S3 after the resume. A finished run has no next nodes.
saved = json.loads(s3.get_object(Bucket=bucket, Key=graph_key)["Body"].read())
print(f"saved status       : {saved['status']}, next nodes: {saved['next_nodes_to_execute']}")

# Checks.
assert resumed_result.status == Status.COMPLETED, f"status: {resumed_result.status}"
assert ran_now == ["manager_approval"], f"these nodes ran again: {ran_now}"
assert [node.node_id for node in resumed_result.execution_order][-1] == "manager_approval"
assert saved["status"] == "completed"

Only the interrupted node ran. The four completed nodes came back from S3 with their status and their results, but a restored `AgentResult` keeps only the message and the stop reason. Its `structured_output` is `None`, so a condition that read it after a restore would not route. Graph totals such as `execution_count` and `accumulated_usage` carried on from the saved values.

The new `manager_approval` Agent started with no messages. When it resumed, Strands copied the first instance's conversation from the saved interrupt context, ran the tool again, and `interrupt()` returned the answer. Now the run is complete and the saved state has no next nodes, so another build with this session ID would start the graph from the beginning.

## Route the manager's answer on a validated status

A payment step now follows `manager_approval`, and it must run only when the manager approved. The obvious condition searches the node's text for "approved". The manager's answer this time is "Not approved: attach the summit agenda and resubmit.", which contains "approved".

The graph below has two nodes: `manager_approval` is the entry point, and its task already contains the review. It resumes the same graph instance, so it needs no session manager: the interrupt state is in that instance's memory. Each of the next two cells runs the graph twice, once until the interrupt and once with the answer, each with its own panes. The first cell routes on text, the second on a validated `ManagerDecision`.

Look for:

- With the text condition, does `payment` run?
- With the validated status, does `payment` run?

The next cell defines the contract, `ManagerDecision`, and the rejection the manager sends.

In [ ]:
class ManagerDecision(BaseModel):
    """The approval node's decision. The payment edge routes on `status`."""

    status: Literal["approved", "rejected"] = Field(description="The manager's decision")
    note: str = Field(description="The manager's note")


review_summary = (
    f"{task}\nReview: the claim meets the policy and the receipts add up, but it is over USD 2,500, "
    "so it needs the manager's approval."
)
rejection = "Not approved: attach the summit agenda and resubmit."

In [ ]:
# The edge condition: a substring of the manager's text.
def text_approved(state):
    """Condition for the edge from manager_approval to payment: a substring of the generated text."""
    if "manager_approval" not in state.results:
        return False
    # str() of the node's AgentResult is the text of its final message.
    return "approved" in str(state.results["manager_approval"].result).lower()


# The nodes: the manager asks for approval, and payment pays.
manager_approval = Agent(
    name="manager_approval",
    model=sonnet,
    tools=[request_manager_approval],
    system_prompt=(
        "You get the manager's decision on an expense claim. Call request_manager_approval once with the claim "
        "ID, the amount, and a two-sentence summary of the review. Then report the manager's decision and note."
    ),
    callback_handler=None,
)

payment = Agent(
    name="payment",
    model=nova,
    system_prompt=(
        "You schedule the reimbursement of an approved expense claim. Write one line with the claim ID and amount."
    ),
    callback_handler=None,
)

# The graph: register each Agent as a node.
builder = GraphBuilder()
builder.add_node(manager_approval, "manager_approval")
builder.add_node(payment, "payment")

# The edge: pay only when the text condition matches.
builder.add_edge("manager_approval", "payment", condition=text_approved)
builder.set_entry_point("manager_approval")

# Limits: bound the run.
builder.set_max_node_executions(4)  # node runs in total
builder.set_execution_timeout(300)  # seconds for the whole run
builder.set_node_timeout(180)  # seconds for each node
graph = builder.build()  # no session manager: the interrupt state stays in this graph object

# Run 1: stream until the graph asks the manager.
print("Run 1: until the graph asks the manager")
panes = LivePanes()
async for event in graph.stream_async(review_summary):
    panes.show(event)
paused = event["result"]  # status INTERRUPTED, with the pending interrupt

# Run 2: answer the interrupt with the manager's rejection.
# One interruptResponse block per pending interrupt: its ID and the answer interrupt() will return.
responses = [
    {
        "interruptResponse": {
            "interruptId": paused.interrupts[0].id,
            "response": rejection,
        },
    },
]
print("Run 2: with the manager's answer")
panes = LivePanes()
async for event in graph.stream_async(responses):
    panes.show(event)
text_decided = event["result"]

# Which nodes ran after the manager's answer.
text_ran = [node.node_id for node in text_decided.execution_order]
print(f"nodes that ran: {' -> '.join(text_ran)} (payment {'ran' if 'payment' in text_ran else 'did not run'})")

The text condition found "approved" inside "Not approved", so `payment` ran for a rejected claim.

The next cell gives `manager_approval` the structured output model `ManagerDecision`, and the edge condition reads its validated `status` instead of the text.

**What to look for:** the same two runs, and this time no `payment` pane.

In [ ]:
# The edge condition: the manager's validated status.
def manager_approved(state):
    """Condition for the edge from manager_approval to payment."""
    if "manager_approval" not in state.results:
        return False
    decision = state.results["manager_approval"].result.structured_output  # the validated ManagerDecision, or None
    return decision is not None and decision.status == "approved"


# The nodes: the manager asks for approval, and payment pays.
manager_approval = Agent(
    name="manager_approval",
    model=sonnet,
    tools=[request_manager_approval],
    structured_output_model=ManagerDecision,  # the typed decision the payment edge reads
    system_prompt=(
        "You get the manager's decision on an expense claim. Call request_manager_approval once with the claim "
        "ID, the amount, and a two-sentence summary of the review. Then report the manager's decision and note."
    ),
    callback_handler=None,
)

payment = Agent(
    name="payment",
    model=nova,
    system_prompt=(
        "You schedule the reimbursement of an approved expense claim. Write one line with the claim ID and amount."
    ),
    callback_handler=None,
)

# The graph: register each Agent as a node.
builder = GraphBuilder()
builder.add_node(manager_approval, "manager_approval")
builder.add_node(payment, "payment")

# The edge: pay only when the validated status is approved.
builder.add_edge("manager_approval", "payment", condition=manager_approved)
builder.set_entry_point("manager_approval")

# Limits: bound the run.
builder.set_max_node_executions(4)  # node runs in total
builder.set_execution_timeout(300)  # seconds for the whole run
builder.set_node_timeout(180)  # seconds for each node
graph = builder.build()  # no session manager: the interrupt state stays in this graph object

# Run 1: stream until the graph asks the manager.
print("Run 1: until the graph asks the manager")
panes = LivePanes()
async for event in graph.stream_async(review_summary):
    panes.show(event)
paused = event["result"]  # status INTERRUPTED, with the pending interrupt

# Run 2: answer the interrupt with the manager's rejection.
# One interruptResponse block per pending interrupt: its ID and the answer interrupt() will return.
responses = [
    {
        "interruptResponse": {
            "interruptId": paused.interrupts[0].id,
            "response": rejection,
        },
    },
]
print("Run 2: with the manager's answer")
panes = LivePanes()
async for event in graph.stream_async(responses):
    panes.show(event)
decided = event["result"]

# Which nodes ran after the manager's answer.
ran = [node.node_id for node in decided.execution_order]
decision = decided.results["manager_approval"].result.structured_output
print(f"nodes that ran: {' -> '.join(ran)} (payment {'ran' if 'payment' in ran else 'did not run'})")
print(f"validated manager decision: {decision}")

# Validate both demonstrations while their results are still in the same cell.
assert "payment" in text_ran, "the text condition did not demonstrate the substring bug"
assert decided.status == Status.COMPLETED, f"status: {decided.status}"
assert decision is not None, "manager_approval did not return its structured output"
assert decision.status == "rejected", f"the decision was {decision.status}"
assert "payment" not in ran, "payment ran for a rejected claim"

### Verify

With the text condition, `payment` ran for a rejected claim. With the validated status it did not, and the rejection is data you can store or send back to the employee.

In [ ]:
print("The preceding run validated both routes: text paid the rejected claim; structured status did not.")

## Clean up

The next cell deletes the lab stack only if this notebook deployed it (its `ProvisionedBy` output is `notebook`). It empties the session bucket first, because CloudFormation cannot delete a bucket that still holds objects. At an event, Workshop Studio owns the stack and the cell leaves it in place.

In [ ]:
cleanup("graph", empty_buckets=[bucket])

In [ ]:
write_validation_artifact("graph", {
    "status": decided.status.value,
    "execution_order": ran,
    "manager_decision": decision.status,
    "payment_ran": "payment" in ran,
})

## Takeaways

In a Graph your code owns the route: nodes, edges, conditions, and limits. The models judge inside their nodes, and `execution_order` and the per-node results show what ran.

Route on a validated field, never on a substring of generated text. A `structured_output_model` on the node gives the edge condition a field it can trust.

An interrupt stops the graph with status `INTERRUPTED` and a result, not an exception. With the session manager on the graph, a new instance with the same session ID resumes the run. Completed nodes do not run again, their results come back as text only, and the interrupted node continues its own conversation.

Amazon Bedrock charged for the model calls and Amazon S3 for a few small objects. Every node is another model call; the resume paid only for the one node that ran.

## Where to go next

The [Graph guide](https://strandsagents.com/docs/user-guide/sdk/multi-agent/graph/) covers cycles with `reset_on_revisit`, nested multi-agent nodes, and custom nodes. The [human-in-the-loop guide](https://strandsagents.com/docs/user-guide/sdk/agents/interventions/human-in-the-loop/) covers interrupts raised from hooks, and the [session management guide](https://strandsagents.com/docs/user-guide/sdk/agents/session-management/) covers the file and S3 session managers.

## Source and license

The lab uses the core `strands.multiagent.Graph` class. An earlier version was adapted from the [Strands Agents Graph sample](https://github.com/strands-agents/samples/tree/main/python/01-learn/12-graph), licensed under Apache License 2.0. The scenario, prompts, code, and text were rewritten for this workshop. See `LICENSE` and `NOTICE`.